In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from custom_transformers_v2 import ResumeFeatureBuilderV2

raw = pd.read_csv("../data/raw/resume_data_for_ranking.csv")
feat_ref = pd.read_parquet("../data/processed/v2_features.parquet")

sample_idx = raw.sample(50, random_state=42).index
builder = ResumeFeatureBuilderV2()
recomputed = builder.transform(raw.loc[sample_idx])
recomputed.index = sample_idx

for col in ["semantic_similarity", "education_similarity", "skills_soft_coverage"]:
    diff = (recomputed[col] - feat_ref.loc[sample_idx, col]).abs()
    print(f"{col:25s} maximum difference : {diff.max():.4f}  (NaN on both sides : {(recomputed[col].isna() & feat_ref.loc[sample_idx, col].isna()).sum()})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

semantic_similarity       maximum difference : 0.0000  (NaN on both sides : 0)
education_similarity      maximum difference : 0.0000  (NaN on both sides : 1)
skills_soft_coverage      maximum difference : 0.0000  (NaN on both sides : 13)


In [2]:
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from custom_transformers_v2 import ResumeFeatureBuilderV2

ridge_estimator = joblib.load("../data/processed/v2_ridge_estimator.pkl")
calibrator = joblib.load("../data/processed/v2_score_calibrator.pkl")

full_pipeline_v2 = Pipeline([
    ("features", ResumeFeatureBuilderV2()),
    ("model", ridge_estimator),
])

raw = pd.read_csv("../data/raw/resume_data_for_ranking.csv")
sample = raw.sample(5, random_state=7)

raw_rank_pred = full_pipeline_v2.predict(sample)
calibrated_pred = calibrator.predict(raw_rank_pred)

for i, (idx, row) in enumerate(sample.iterrows()):
    print(f"actual score={row['matched_score']:.3f}  |  calibrated v2 score={calibrated_pred[i]:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

actual score=0.603  |  calibrated v2 score=0.603
actual score=0.717  |  calibrated v2 score=0.759
actual score=0.917  |  calibrated v2 score=0.603
actual score=0.693  |  calibrated v2 score=0.659
actual score=0.480  |  calibrated v2 score=0.603


In [3]:
problem_row = sample.iloc[[2]]  # the row with an actual score of 0.917

features_only = ResumeFeatureBuilderV2().transform(problem_row)
print(features_only)

print("\njob_position_name :", problem_row["job_position_name"].values[0])
print("career_objective present :", pd.notna(problem_row["career_objective"].values[0]))
print("skills :", problem_row["skills"].values[0])
print("skills_required :", problem_row["skills_required"].values[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   semantic_similarity  education_similarity  skills_soft_coverage
0             0.062603              0.633546              0.142652

job_position_name : Senior iOS Engineer
career_objective present : False
skills : ['Game Theory', 'Bounded Rationality', 'Cryptography', 'Algorithms', 'Social Networks', 'Analysis of Algorithms', 'Combinatorial Optimizations']
skills_required : iOS
iOS App Developer
iOS Application Development
iOS Development
Mobile apps Developer (iOS)
Native IOS
Swift (iOS)
Swift UI


In [4]:
test_sample = full_pipeline_v2.predict(raw.sample(500, random_state=1))
calibrated_sample = calibrator.predict(test_sample)
real_sample_idx = raw.sample(500, random_state=1).index
gap = (raw.loc[real_sample_idx, "matched_score"].values - calibrated_sample)

print(f"Mean absolute difference : {abs(gap).mean():.3f}")
print(f"% of rows with a difference > 0.3 : {(abs(gap) > 0.3).mean():.1%}")

Mean absolute difference : 0.121
% of rows with a difference > 0.3 : 6.2%


In [5]:
from sklearn.base import BaseEstimator, RegressorMixin

class CalibratedRankPipeline(BaseEstimator, RegressorMixin):
    """Chains ResumeFeatureBuilderV2 -> Ridge (rank) -> isotonic calibrator (score 0-1).
    A single object: .predict(raw DataFrame) -> calibrated score."""

    def __init__(self, feature_builder, rank_estimator, calibrator):
        self.feature_builder = feature_builder
        self.rank_estimator = rank_estimator
        self.calibrator = calibrator

    def fit(self, X, y=None):
        return self  # The 3 components are already trained separately (steps 5 and 8).

    def predict(self, X):
        features = self.feature_builder.transform(X)
        raw_rank = self.rank_estimator.predict(features)
        return self.calibrator.predict(raw_rank)

In [7]:
import os 
from custom_transformers_v2 import CalibratedRankPipeline

full_pipeline_v2 = CalibratedRankPipeline(
    feature_builder=ResumeFeatureBuilderV2(),
    rank_estimator=ridge_estimator,
    calibrator=calibrator,
)

joblib.dump(full_pipeline_v2, "../data/processed/full_pipeline_v2.pkl")

size_bytes = os.path.getsize("../data/processed/full_pipeline_v2.pkl")
print(f"Taille : {size_bytes} octets ({size_bytes / 1024:.2f} Ko)")

Taille : 3174 octets (3.10 Ko)


In [8]:
import subprocess
import sys

script = """
import sys
sys.path.append('../src')
import joblib
import pandas as pd

pipeline = joblib.load('../data/processed/full_pipeline_v2.pkl')
raw = pd.read_csv('../data/raw/resume_data_for_ranking.csv')
sample = raw.sample(3, random_state=7)
print(pipeline.predict(sample))
"""

result = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

STDOUT: [0.60284714 0.75871387 0.60284714]

STDERR: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Loading weights: 100%|##########| 103/103 [00:00<00:00, 1731.43it/s]

